# CRISP-DM Stage 2: Data Preparation

1. **Clean Data & Regional Imputation** (Median Imputation & Geo-Merge)
2. **Transform Data** (Log Natural $\ln(1+x)$ & Z-Score `StandardScaler`)

In [ ]:
# Parameter Injeksi (DVC / Papermill)
log_transform_features = [
    'total_koperasi',
    'simpanan_pokok',
    'simpanan_wajib',
    'volume_transaksi',
    'nilai_transaksi'
]


## 1. Clean Data & Spatial Geo-Merge

In [ ]:
import os, json, pandas as pd, numpy as np
from config import (
    RAW_PROVINCES_CSV, RAW_REGENCIES_CSV, GEO_PROVINCES_JSON, GEO_REGENCIES_JSON,
    CLEANED_PROVINCES_CSV, CLEANED_REGENCIES_CSV, NUMERIC_COLUMNS, FEATURE_COLUMNS,
    SCALED_FEATURES_CSV, PREPARATION_META_JSON
)

# 1. Clean Provinces
df_p = pd.read_csv(RAW_PROVINCES_CSV)
df_p['province_name'] = df_p['province_name'].astype(str).str.strip().str.upper()

if os.path.exists(GEO_PROVINCES_JSON):
    with open(GEO_PROVINCES_JSON, encoding='utf-8') as f:
        geo_p = pd.DataFrame(json.load(f))
    if not geo_p.empty:
        geo_p['province_name_clean'] = geo_p['name'].astype(str).str.strip().str.upper()
        df_p = df_p.merge(geo_p[['province_name_clean', 'province_id', 'latitude', 'longitude']], left_on='province_name', right_on='province_name_clean', how='left').drop(columns=['province_name_clean'], errors='ignore')

df_p['rasio_nib'] = np.where(df_p['total_koperasi'] > 0, (df_p['koperasi_nib'] / df_p['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_p['rasio_npwp'] = np.where(df_p['total_koperasi'] > 0, (df_p['koperasi_npwp'] / df_p['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_p['rasio_rat'] = np.where(df_p['total_koperasi'] > 0, (df_p['koperasi_rat'] / df_p['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)

os.makedirs(os.path.dirname(CLEANED_PROVINCES_CSV), exist_ok=True)
df_p.to_csv(CLEANED_PROVINCES_CSV, index=False)

# 2. Clean Regencies
df_r = pd.read_csv(RAW_REGENCIES_CSV)
df_r['regency_name'] = df_r['regency_name'].astype(str).str.strip().str.upper()

for col in NUMERIC_COLUMNS:
    if col in df_r.columns:
        df_r[col] = df_r.groupby('province_id')[col].transform(lambda s: s.fillna(s.median() if not s.dropna().empty else 0)).fillna(0)

if os.path.exists(GEO_REGENCIES_JSON):
    with open(GEO_REGENCIES_JSON, encoding='utf-8') as f:
        geo_r = pd.DataFrame(json.load(f))
    if not geo_r.empty:
        df_r = df_r.merge(geo_r[['province_id', 'regency_no', 'latitude', 'longitude']], on=['province_id', 'regency_no'], how='left')

df_r['rasio_nib'] = np.where(df_r['total_koperasi'] > 0, (df_r['koperasi_nib'] / df_r['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_r['rasio_npwp'] = np.where(df_r['total_koperasi'] > 0, (df_r['koperasi_npwp'] / df_r['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_r['rasio_rat'] = np.where(df_r['total_koperasi'] > 0, (df_r['koperasi_rat'] / df_r['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)

os.makedirs(os.path.dirname(CLEANED_REGENCIES_CSV), exist_ok=True)
df_r.to_csv(CLEANED_REGENCIES_CSV, index=False)
print(f"Cleaned {len(df_p)} provinsi dan {len(df_r)} kabupaten/kota.")


## 2. Transform Data (Log-Transform & Z-Score Scaling)

In [ ]:
from sklearn.preprocessing import StandardScaler

features_present = [col for col in FEATURE_COLUMNS if col in df_r.columns]
X_df = df_r[features_present].copy().fillna(0)

# Log Transformation
for col in log_transform_features:
    if col in X_df.columns:
        X_df[col] = np.log1p(np.maximum(X_df[col].values, 0))

# StandardScaler Normalization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values)

scaled_df = pd.DataFrame(X_scaled, columns=[f"scaled_{c}" for c in features_present])
os.makedirs(os.path.dirname(SCALED_FEATURES_CSV), exist_ok=True)
scaled_df.to_csv(SCALED_FEATURES_CSV, index=False)

meta = {
    "total_samples": len(df_r), "total_features": len(features_present),
    "feature_columns": features_present, "log_transformed_features": [c for c in log_transform_features if c in features_present],
    "scaler_type": "StandardScaler (Z-Score Normalization)"
}
with open(PREPARATION_META_JSON, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
print("Transformation & Scaling completed successfully.")
